# 🎭 Face Swap Live — Google Colab (GPU)

**ANTES DE QUALQUER COISA:**
1. `Runtime → Change runtime type → T4 GPU → Save`
2. `Runtime → Run all` (ou execute célula por célula em ordem)

Se mudou o runtime depois de abrir, faça `Runtime → Disconnect and delete runtime` e abra de novo.

In [ ]:
# Célula 1 — Verifica GPU e CUDA
import subprocess

r = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if r.returncode != 0:
    print('ERRO: GPU nao encontrada!')
    print('Va em Runtime -> Change runtime type -> T4 GPU e reinicie tudo.')
else:
    print(r.stdout)
    import onnxruntime as ort
    providers = ort.get_available_providers()
    print('Providers ONNX disponíveis:', providers)
    if 'CUDAExecutionProvider' not in providers:
        print()
        print('AVISO: CUDA nao disponivel no onnxruntime ainda.')
        print('Isso pode acontecer se onnxruntime-gpu foi instalado com runtime CPU.')
        print('Solucao: Runtime -> Disconnect and delete runtime -> mudar para T4 -> rodar de novo.')
    else:
        print('CUDA disponivel! Pronto para GPU.')

In [ ]:
# Célula 2 — Instala dependências
!pip install -q flask flask-socketio eventlet \
    insightface onnxruntime-gpu opencv-python-headless \
    huggingface_hub pillow numpy

# cloudflared: tunnel público sem precisar de conta
import os
if not os.path.exists('/usr/local/bin/cloudflared'):
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
    !chmod +x /usr/local/bin/cloudflared
    print('cloudflared instalado.')
else:
    print('cloudflared ja existe.')

print('Instalacao concluida!')

In [ ]:
# Célula 3 — Clona / atualiza o repositório (caminho fixo, sem duplicar)
import os

REPO = '/content/faceSwapLive2'

if not os.path.exists(REPO):
    !git clone https://github.com/moniquepavan/faceSwapLive2.git {REPO}
else:
    !git -C {REPO} pull

os.chdir(REPO)
print('Diretorio atual:', os.getcwd())
print('Arquivos:', os.listdir('.'))

In [ ]:
# Célula 4 — Baixa o modelo inswapper (~280 MB)
import os, shutil
from huggingface_hub import hf_hub_download

DEST = '/content/faceSwapLive2/models/inswapper_128.onnx'
os.makedirs('/content/faceSwapLive2/models', exist_ok=True)

if os.path.exists(DEST):
    print(f'Modelo ja existe ({os.path.getsize(DEST)/1e6:.0f} MB) — pulando.')
else:
    token = input('Cole seu token do Hugging Face (huggingface.co/settings/tokens): ').strip()
    print('Baixando...')
    path = hf_hub_download(
        repo_id='hacksider/deep-live-cam',
        filename='inswapper_128_fp16.onnx',
        token=token,
        local_dir='/content/faceSwapLive2/models'
    )
    if os.path.abspath(path) != os.path.abspath(DEST):
        shutil.move(path, DEST)
    print(f'Download concluido! {os.path.getsize(DEST)/1e6:.0f} MB')

In [ ]:
# Célula 5 — Inicia servidor Flask + tunnel cloudflared
import subprocess, threading, time, sys, re, os

os.chdir('/content/faceSwapLive2')

# Para processos anteriores se houver
subprocess.run(['pkill', '-f', 'app_colab.py'], capture_output=True)
subprocess.run(['pkill', '-f', 'cloudflared'],  capture_output=True)
time.sleep(1)

# Inicia Flask
server = subprocess.Popen(
    [sys.executable, 'app_colab.py'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    cwd='/content/faceSwapLive2'
)

def print_server_logs():
    for line in server.stdout:
        print('[srv]', line.rstrip())
threading.Thread(target=print_server_logs, daemon=True).start()

print('Aguardando Flask iniciar (6s)...')
time.sleep(6)

# Inicia tunnel cloudflared (sem conta, sem token)
cf = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://localhost:5000'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

print('Aguardando URL do tunnel (pode levar ~15s)...')
url = None
for line in cf.stdout:
    line = line.rstrip()
    print('[cf]', line)
    match = re.search(r'https://[\w\-]+\.trycloudflare\.com', line)
    if match:
        url = match.group(0)
        break

print()
print('=' * 60)
print(f'  ABRA NO NAVEGADOR: {url}')
print('=' * 60)
print()
print('Aguarde o status "Pronto! [CUDA GPU]" aparecer (~60s).')
print('Mantenha esta celula rodando. Ctrl+C para encerrar.')

try:
    cf.wait()
except KeyboardInterrupt:
    cf.terminate()
    server.terminate()
    print('Encerrado.')